In [ ]:
import pandas as pd
import requests

# ====================================================
# --- 1️⃣ Descargar datos de inflación de Eurostat (EA20, All-items) ---
# ====================================================
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/prc_hicp_manr"

params = {
    "geo": "EA20",   # Eurozona 20 países
    "coicop": "CP00",  # All-items HICP
    "unit": "RCH_A"    # tasa anual de cambio (%)
}

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()

# ====================================================
# --- 2️⃣ Extraer fechas y valores ---
# ====================================================
# Fechas (etiquetas mensuales)
time_labels = data["dimension"]["time"]["category"]["label"]
dates = list(time_labels.keys())  # formato '2000-01', '2000-02', ...

# Valores (diccionario {índice: valor})
values_dict = data["value"]

# Convertir a DataFrame seguro
df = pd.DataFrame({"date": pd.to_datetime(dates, format="%Y-%m")})
df["inflation"] = None

# Rellenar solo los índices que existen en 'values_dict'
for i, val in values_dict.items():
    if int(i) < len(df):
        df.loc[int(i), "inflation"] = val

# ====================================================
# --- 3️⃣ Limpieza final ---
# ====================================================
df["inflation"] = pd.to_numeric(df["inflation"], errors="coerce")
df = df.sort_values("date").dropna(subset=["inflation"])
df = df[df["date"].dt.year >= 2000]  # desde 2000 en adelante

# ====================================================
# --- 4️⃣ (Opcional) Convertir a diario ---
# ====================================================
to_daily = True  # cambia a False si prefieres mantener mensual

if to_daily:
    daily_index = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="B")
    df_daily = df.set_index("date").reindex(daily_index, method="ffill")
    df_daily = df_daily.rename_axis("date").reset_index()
    df = df_daily

# ====================================================
# --- 5️⃣ Guardar y mostrar ---
# ====================================================
df.to_csv("../data/csv/raw/euro_inflation_2000_2025_real.csv", index=False)
print(f"✅ Datos de inflación descargados y guardados hasta {df['date'].max().date()}")
print(df.tail(10))


✅ Datos de inflación descargados y guardados hasta 2025-09-01
           date  inflation
6447 2025-08-19        2.1
6448 2025-08-20        2.1
6449 2025-08-21        2.1
6450 2025-08-22        2.1
6451 2025-08-25        2.1
6452 2025-08-26        2.1
6453 2025-08-27        2.1
6454 2025-08-28        2.1
6455 2025-08-29        2.1
6456 2025-09-01        2.2


In [1]:
import pandas as pd

# ============================================================
# 🔧 CONFIGURACIÓN
# ============================================================
valor_inflacion = 2.2   # Inflación mensual (%) para sep-oct 2025
inicio = "2025-09-01"
fin = "2025-10-31"

# ============================================================
# 📅 Generar calendario de días hábiles
# ============================================================
# freq="B" => solo Business Days (lunes a viernes)
fechas = pd.date_range(start=inicio, end=fin, freq="B")

# Crear DataFrame
df_extendido = pd.DataFrame({
    "date": fechas,
    "inflation": valor_inflacion
})

# Mostrar resultado
print(df_extendido.head(10))
print("...")
print(df_extendido.tail(10))

# ============================================================
# 💾 Guardar en CSV (listo para añadir a tu dataset)
# ============================================================
df_extendido.to_csv("../data/csv/raw/inflation_sep_oct_2025.csv", index=False)

print("\n✅ CSV generado correctamente con inflación = 2.4% para días hábiles de septiembre y octubre 2025.")


        date  inflation
0 2025-09-01        2.2
1 2025-09-02        2.2
2 2025-09-03        2.2
3 2025-09-04        2.2
4 2025-09-05        2.2
5 2025-09-08        2.2
6 2025-09-09        2.2
7 2025-09-10        2.2
8 2025-09-11        2.2
9 2025-09-12        2.2
...
         date  inflation
35 2025-10-20        2.2
36 2025-10-21        2.2
37 2025-10-22        2.2
38 2025-10-23        2.2
39 2025-10-24        2.2
40 2025-10-27        2.2
41 2025-10-28        2.2
42 2025-10-29        2.2
43 2025-10-30        2.2
44 2025-10-31        2.2

✅ CSV generado correctamente con inflación = 2.4% para días hábiles de septiembre y octubre 2025.


In [2]:
import pandas as pd

# ============================================================
# 🔧 RUTAS DE ARCHIVOS
# ============================================================
ruta_principal = r"C:\Users\josit\CUARTO CURSO\APRENDIZAJE AUTOMATICO\Caso02_Prediccion_BBVA_SANTANDER\data\csv\raw\euro_inflation_2000_2025_real.csv"
ruta_extendida = r"C:\Users\josit\CUARTO CURSO\APRENDIZAJE AUTOMATICO\Caso02_Prediccion_BBVA_SANTANDER\data\csv\raw\inflation_sep_oct_2025.csv"
ruta_salida = r"C:\Users\josit\CUARTO CURSO\APRENDIZAJE AUTOMATICO\Caso02_Prediccion_BBVA_SANTANDER\data\csv\clean\euro_inflation_2000_2025_final.csv"

# ============================================================
# 📂 1️⃣ Cargar los dos datasets
# ============================================================
df_base = pd.read_csv(ruta_principal)
df_ext = pd.read_csv(ruta_extendida)

# Asegurar formato de fecha correcto
df_base["date"] = pd.to_datetime(df_base["date"])
df_ext["date"] = pd.to_datetime(df_ext["date"])

# ============================================================
# 🔄 2️⃣ Fusionar ambos y limpiar duplicados
# ============================================================
df_fusionado = pd.concat([df_base, df_ext], ignore_index=True)
df_fusionado = df_fusionado.drop_duplicates(subset="date", keep="last")
df_fusionado = df_fusionado.sort_values("date").reset_index(drop=True)

# ============================================================
# 💾 3️⃣ Guardar CSV final
# ============================================================
df_fusionado.to_csv(ruta_salida, index=False)

# ============================================================
# ✅ 4️⃣ Confirmación
# ============================================================
print(f"✅ Dataset fusionado correctamente.")
print(f"Total de filas: {len(df_fusionado)}")
print(f"Rango temporal: {df_fusionado['date'].min().date()} → {df_fusionado['date'].max().date()}")
print(f"\nGuardado en:\n{ruta_salida}")


✅ Dataset fusionado correctamente.
Total de filas: 6740
Rango temporal: 2000-01-03 → 2025-10-31

Guardado en:
C:\Users\josit\CUARTO CURSO\APRENDIZAJE AUTOMATICO\Caso02_Prediccion_BBVA_SANTANDER\data\csv\clean\euro_inflation_2000_2025_final.csv


In [ ]:
import pandas as pd

# ============================================================
# 🔧 RUTAS
# ============================================================
ruta_bce = "../data/csv/raw/ECB Data Portal_20251105153445.csv"
ruta_salida = "../data/csv/clean/euro_interest_2000_2025_clean.csv"

# ============================================================
# 📂 1️⃣ Cargar datos originales
# ============================================================
df = pd.read_csv(ruta_bce)

# Normalizamos los nombres de columna
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Detectar columnas importantes
fecha_col = "date"
deposit_col = [c for c in df.columns if "deposit" in c][0]
marginal_col = [c for c in df.columns if "marginal" in c][0]
refinancing_col = [c for c in df.columns if "main_refinancing" in c or "mrr" in c][0]

# ============================================================
# 🧹 2️⃣ Limpiar y filtrar fechas
# ============================================================
df = df[[fecha_col, deposit_col, marginal_col, refinancing_col]].copy()
df[fecha_col] = pd.to_datetime(df[fecha_col], errors="coerce")
df = df.dropna(subset=[fecha_col])
df = df.sort_values(fecha_col)

# Convertir valores a numéricos
for col in [deposit_col, marginal_col, refinancing_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Filtrar rango 2000 → 31/10/2025
mask = (df[fecha_col] >= "2000-01-01") & (df[fecha_col] <= "2025-10-31")
df = df.loc[mask].reset_index(drop=True)

# ============================================================
# 📅 3️⃣ (Opcional) Eliminar fines de semana
# ============================================================
df = df[df[fecha_col].dt.dayofweek < 5]  # 0=lunes, 6=domingo

# ============================================================
# 💾 4️⃣ Renombrar y guardar CSV limpio
# ============================================================
df = df.rename(columns={
    deposit_col: "deposit_rate",
    marginal_col: "marginal_rate",
    refinancing_col: "refinancing_rate"
})

df.to_csv(ruta_salida, index=False)

# ============================================================
# ✅ 5️⃣ Confirmación
# ============================================================
print("✅ Tipos de interés del BCE procesados correctamente.")
print(f"Total de filas: {len(df)}")
print(f"Rango temporal: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"\nGuardado en:\n{ruta_salida}")
print("\nVista previa:")
print(df.head(10))
